In [ ]:
import jax
import jax.numpy as jnp
import jax.tree_util as jtu
import numpy as np

import timeit
import imageio
import matplotlib.pyplot as plt
from tqdm.auto import trange, tqdm
import os
from xminigrid.types import RuleSet
from xminigrid.benchmarks import Benchmark, load_benchmark, load_benchmark_from_path, load_bz2_pickle, DATA_PATH, NAME2HFFILENAME
from xminigrid.rendering.text_render import print_ruleset
# utils for the demonstation
from xminigrid.core.grid import room
from xminigrid.types import AgentState
from xminigrid.core.actions import take_action
from xminigrid.core.constants import Tiles, Colors, TILES_REGISTRY
from xminigrid.rendering.rgb_render import render

# rules and goals
from xminigrid.core.goals import check_goal, AgentNearGoal
from xminigrid.core.rules import check_rule, AgentNearRule
def show_img(img, dpi=32):
    plt.figure(dpi=dpi)
    plt.axis('off')
    plt.imshow(img)
    plt.show()  # 在 Notebook 中显示图像

import xminigrid
i_indices = jnp.arange(9)
j_indices = jnp.arange(9)
i_grid, j_grid = jnp.meshgrid(i_indices, j_indices, indexing='ij')

# 展平 i_grid 和 j_grid，准备并行化处理
i_grid_flat = i_grid.flatten()
j_grid_flat = j_grid.flatten()
up_x = i_grid_flat-8
up_y = j_grid_flat-4
right_x = j_grid_flat-4
right_y = -(i_grid_flat-8)
down_x = -(i_grid_flat-8)
down_y = -(j_grid_flat-4)
left_x = -(j_grid_flat-4)
left_y = i_grid_flat-8

goal = AgentNearGoal(tile=TILES_REGISTRY[Tiles.SQUARE, Colors.PURPLE])
rule = AgentNearRule(
    tile=TILES_REGISTRY[Tiles.BALL, Colors.YELLOW], 
    prod_tile=TILES_REGISTRY[Tiles.SQUARE, Colors.PURPLE],
)

ruleset = RuleSet(
    goal=goal.encode(),
    rules=rule.encode()[None, ...],
    init_tiles=jnp.array((
        TILES_REGISTRY[Tiles.BALL, Colors.YELLOW],
    ))
)
print_ruleset(ruleset)
import jax
import jax.numpy as jnp
import jax.tree_util as jtu
from xminigrid.wrappers import GymAutoResetWrapper
import imageio
# 创建 rollout 函数
def build_rollout(env, env_params, num_steps):
    def rollout(rng):
        def _step_fn(carry, _):
            rng, timestep = carry
            rng, _rng = jax.random.split(rng)
            action = jax.random.randint(_rng, shape=(), minval=0, maxval=env.num_actions(env_params))
            
            timestep = env.step(env_params, timestep, action)
            
            return (rng, timestep), timestep
    
        rng, _rng = jax.random.split(rng)
        timestep = env.reset(env_params, _rng)
        
        rng, transitions = jax.lax.scan(_step_fn, (rng, timestep), None, length=num_steps)
        return transitions

    return rollout

# 创建环境并包裹自动重置
env, env_params = xminigrid.make("MiniGrid-EmptyRandom-8x8")
env = GymAutoResetWrapper(env)

# 设置步数并进行 JIT 编译
num_steps = 10  # 设定要执行的步数
rollout_fn = jax.jit(build_rollout(env, env_params, num_steps=num_steps))

# 执行 rollout 并记录结果
transitions = rollout_fn(jax.random.PRNGKey(0))

# 打印结果的形状信息
print("Transitions shapes: \n", jtu.tree_map(jnp.shape, transitions))
images = []

for i in trange(1000):
    timestep = jtu.tree_map(lambda x: x[i], transitions)
    images.append(env.render(env_params, timestep))

imageio.mimsave("example_rollout.mp4", images, fps=32, format="mp4")
breakpoint()
from IPython.display import Video

Video("example_rollout.mp4", embed=True)
breakpoint()

from jax import jit
@jit
def _is_in_bound(x,y):
    return (x >= 0) & (x <= 8) & (y >= 0) & (y <= 8)
@jit   
def process_batch(batch,dir,pos):
    

    def case_0():
        local_obs = jnp.zeros((9, 9, 2), dtype=jnp.uint8)
        x = up_x + pos[0]
        y = up_y + pos[1]
        mask = _is_in_bound(x, y)

        # 遍历每个位置，仅在满足条件的 (x, y) 位置上更新
        def update_local_obs(i, obs):
            xi, yi = x[i], y[i]

            def set_update_value(obs):
                update_value = batch[xi - pos[0] + 8, yi - pos[1] + 4]
                return obs.at[xi, yi, :].set(update_value)

            # 使用 jax.lax.cond 进行条件更新
            obs = jax.lax.cond(
                mask[i],           # 条件为 True 时更新
                set_update_value,   # 满足条件时的更新函数
                lambda obs: obs,    # 不满足条件时保持不变
                obs                 # 传递的数组
            )
            return obs

        # 遍历 x 和 y 坐标，并逐个更新 local_obs
        for i in range(len(x)):
            local_obs = update_local_obs(i, local_obs)
        
        return local_obs
    
    def case_1():
        local_obs = jnp.zeros((9, 9, 2), dtype=jnp.uint8)
        x = right_x + pos[0]
        y = right_y + pos[1]
        mask = _is_in_bound(x, y)

        # 遍历每个位置，仅在满足条件的 (x, y) 位置上更新
        def update_local_obs(i, obs):
            xi, yi = x[i], y[i]

            def set_update_value(obs):
                update_value = batch[8 + pos[1] - yi, xi + 4 - pos[0]]
                return obs.at[xi, yi, :].set(update_value)

            # 使用 jax.lax.cond 进行条件更新
            obs = jax.lax.cond(mask[i], set_update_value, lambda obs: obs, obs)
            return obs

        for i in range(len(x)):
            local_obs = update_local_obs(i, local_obs)
        
        return local_obs

    def case_2():
        local_obs = jnp.zeros((9, 9, 2), dtype=jnp.uint8)
        x = down_x + pos[0]
        y = down_y + pos[1]
        mask = _is_in_bound(x, y)

        # 遍历每个位置，仅在满足条件的 (x, y) 位置上更新
        def update_local_obs(i, obs):
            xi, yi = x[i], y[i]

            def set_update_value(obs):
                update_value = batch[8 + pos[0] - xi, 4 + pos[1] - yi]
                return obs.at[xi, yi, :].set(update_value)
            obs = jax.lax.cond(mask[i], set_update_value, lambda obs: obs, obs)
            return obs

        for i in range(len(x)):
            local_obs = update_local_obs(i, local_obs)
        
        return local_obs
    
    def case_3():
        local_obs = jnp.zeros((9, 9, 2), dtype=jnp.uint8)
        x = left_x + pos[0]
        y = left_y + pos[1]
        mask = _is_in_bound(x, y)

        # 遍历每个位置，仅在满足条件的 (x, y) 位置上更新
        def update_local_obs(i, obs):
            xi, yi = x[i], y[i]

            def set_update_value(obs):
                update_value = batch[8 + yi - pos[1], 4 - xi + pos[0]]
                return obs.at[xi, yi, :].set(update_value)

            obs = jax.lax.cond(mask[i], set_update_value, lambda obs: obs, obs)
            return obs

        for i in range(len(x)):
            local_obs = update_local_obs(i, local_obs)
        
        return local_obs
    local_obs_final = jax.lax.switch(
        dir,
        [case_0, case_1, case_2, case_3]
    )
    return local_obs_final
print("Observation shape:", timestep.observation.shape)


all_batches_label_obs = process_batch(
    timestep.observation, 
    timestep.state.agent.direction.astype(int),
    timestep.state.agent.position
)



timestep = timestep.replace(observation = all_batches_label_obs)

# 对于批次中的每个样本，使用 jax.lax.fori_loop



: 

import jax
import jax.numpy as jnp
import jax.tree_util as jtu
import numpy as np

import timeit
import imageio
import matplotlib.pyplot as plt
from tqdm.auto import trange, tqdm
import os
from xminigrid.types import RuleSet
from xminigrid.benchmarks import Benchmark, load_benchmark, load_benchmark_from_path, load_bz2_pickle, DATA_PATH, NAME2HFFILENAME
from xminigrid.rendering.text_render import print_ruleset
# utils for the demonstation
from xminigrid.core.grid import room
from xminigrid.types import AgentState
from xminigrid.core.actions import take_action
from xminigrid.core.constants import Tiles, Colors, TILES_REGISTRY
from xminigrid.rendering.rgb_render import render

# rules and goals
from xminigrid.core.goals import check_goal, AgentNearGoal
from xminigrid.core.rules import check_rule, AgentNearRule
def show_img(img, dpi=32):
    plt.figure(dpi=dpi)
    plt.axis('off')
    plt.imshow(img)
    plt.show()  # 在 Notebook 中显示图像

import xminigrid
i_indices = jnp.arange(9)
j_indices = jnp.arange(9)
i_grid, j_grid = jnp.meshgrid(i_indices, j_indices, indexing='ij')


In [ ]:
# 展平 i_grid 和 j_grid，准备并行化处理
i_grid_flat = i_grid.flatten()
j_grid_flat = j_grid.flatten()
up_x = i_grid_flat-8
up_y = j_grid_flat-4
right_x = j_grid_flat-4
right_y = -(i_grid_flat-8)
down_x = -(i_grid_flat-8)
down_y = -(j_grid_flat-4)
left_x = -(j_grid_flat-4)
left_y = i_grid_flat-8

In [ ]:
goal = AgentNearGoal(tile=TILES_REGISTRY[Tiles.SQUARE, Colors.PURPLE])
rule = AgentNearRule(
    tile=TILES_REGISTRY[Tiles.BALL, Colors.YELLOW], 
    prod_tile=TILES_REGISTRY[Tiles.SQUARE, Colors.PURPLE],
)

ruleset = RuleSet(
    goal=goal.encode(),
    rules=rule.encode()[None, ...],
    init_tiles=jnp.array((
        TILES_REGISTRY[Tiles.BALL, Colors.YELLOW],
    ))
)
print_ruleset(ruleset)
import jax
import jax.numpy as jnp
import jax.tree_util as jtu
from xminigrid.wrappers import GymAutoResetWrapper
import imageio

In [ ]:
# 创建 rollout 函数
def build_rollout(env, env_params, num_steps):
    def rollout(rng):
        def _step_fn(carry, _):
            rng, timestep = carry
            rng, _rng = jax.random.split(rng)
            action = jax.random.randint(_rng, shape=(), minval=0, maxval=env.num_actions(env_params))
            
            timestep = env.step(env_params, timestep, action)
            
            return (rng, timestep), timestep
    
        rng, _rng = jax.random.split(rng)
        timestep = env.reset(env_params, _rng)
        
        rng, transitions = jax.lax.scan(_step_fn, (rng, timestep), None, length=num_steps)
        return transitions

    return rollout


In [ ]:
# 创建环境并包裹自动重置
env, env_params = xminigrid.make("XLand-MiniGrid-R1-9x9",view_size=9)
env_params = env_params.replace(ruleset=ruleset)
env = GymAutoResetWrapper(env)

# 设置步数并进行 JIT 编译
num_steps = 10  # 设定要执行的步数
rollout_fn = jax.jit(build_rollout(env, env_params, num_steps=num_steps))

# 执行 rollout 并记录结果
transitions = rollout_fn(jax.random.PRNGKey(0))

# 打印结果的形状信息
print("Transitions shapes: \n", jtu.tree_map(jnp.shape, transitions))
images = []

for i in trange(10):
    timestep = jtu.tree_map(lambda x: x[i], transitions)
    images.append(env.render(env_params, timestep))

imageio.mimsave("example_rollout.mp4", images, fps=1, format="mp4")

from IPython.display import Video

Video("example_rollout.mp4", embed=True)

In [ ]:
from jax import jit
@jit
def _is_in_bound(x,y):
    return (x >= 0) & (x <= 8) & (y >= 0) & (y <= 8)
@jit   
def process_batch(batch,dir,pos):
    

    def case_0():
        local_obs = jnp.zeros((9, 9, 2), dtype=jnp.uint8)
        x = up_x + pos[0]
        y = up_y + pos[1]
        mask = _is_in_bound(x, y)

        # 遍历每个位置，仅在满足条件的 (x, y) 位置上更新
        def update_local_obs(i, obs):
            xi, yi = x[i], y[i]

            def set_update_value(obs):
                update_value = batch[xi - pos[0] + 8, yi - pos[1] + 4]
                return obs.at[xi, yi, :].set(update_value)

            # 使用 jax.lax.cond 进行条件更新
            obs = jax.lax.cond(
                mask[i],           # 条件为 True 时更新
                set_update_value,   # 满足条件时的更新函数
                lambda obs: obs,    # 不满足条件时保持不变
                obs                 # 传递的数组
            )
            return obs

        # 遍历 x 和 y 坐标，并逐个更新 local_obs
        for i in range(len(x)):
            local_obs = update_local_obs(i, local_obs)
        
        return local_obs
    def case_1():
        local_obs = jnp.zeros((9, 9, 2), dtype=jnp.uint8)
        x = right_x + pos[0]
        y = right_y + pos[1]
        mask = _is_in_bound(x, y)

        # 遍历每个位置，仅在满足条件的 (x, y) 位置上更新
        def update_local_obs(i, obs):
            xi, yi = x[i], y[i]

            def set_update_value(obs):
                update_value = batch[8 + pos[1] - yi, xi + 4 - pos[0]]
                return obs.at[xi, yi, :].set(update_value)

            # 使用 jax.lax.cond 进行条件更新
            obs = jax.lax.cond(mask[i], set_update_value, lambda obs: obs, obs)
            return obs

        for i in range(len(x)):
            local_obs = update_local_obs(i, local_obs)
        
        return local_obs
    def case_2():
        local_obs = jnp.zeros((9, 9, 2), dtype=jnp.uint8)
        x = down_x + pos[0]
        y = down_y + pos[1]
        mask = _is_in_bound(x, y)

        # 遍历每个位置，仅在满足条件的 (x, y) 位置上更新
        def update_local_obs(i, obs):
            xi, yi = x[i], y[i]

            def set_update_value(obs):
                update_value = batch[8 + pos[0] - xi, 4 + pos[1] - yi]
                return obs.at[xi, yi, :].set(update_value)
            obs = jax.lax.cond(mask[i], set_update_value, lambda obs: obs, obs)
            return obs

        for i in range(len(x)):
            local_obs = update_local_obs(i, local_obs)
        
        return local_obs
    def case_3():
        local_obs = jnp.zeros((9, 9, 2), dtype=jnp.uint8)
        x = left_x + pos[0]
        y = left_y + pos[1]
        mask = _is_in_bound(x, y)

        # 遍历每个位置，仅在满足条件的 (x, y) 位置上更新
        def update_local_obs(i, obs):
            xi, yi = x[i], y[i]

            def set_update_value(obs):
                update_value = batch[8 + yi - pos[1], 4 - xi + pos[0]]
                return obs.at[xi, yi, :].set(update_value)

            obs = jax.lax.cond(mask[i], set_update_value, lambda obs: obs, obs)
            return obs

        for i in range(len(x)):
            local_obs = update_local_obs(i, local_obs)
        
        return local_obs
    local_obs_final = jax.lax.switch(
        dir,
        [case_0, case_1, case_2, case_3]
    )
    return local_obs_final
print("Observation shape:", timestep.observation.shape)

print("timestep shape:",timestep.shape)
all_batches_label_obs = process_batch(
    timestep.observation, 
    timestep.state.agent.direction.astype(int),
    timestep.state.agent.position
)



timestep = timestep.replace(observation = all_batches_label_obs)